# Numerical Precision

In this section, you will investigate how different convolution
and matrix-matrix multiplication kernel performs when changing the
numerical precision.

## 1. Set-up

In [1]:
# Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
# Make sure your token is stored in a txt file at the location below.
# This way there is no risk that you will push it to your repo
# Never share your token with anyone, it is basically your github password!
with open('/content/gdrive/MyDrive/ece5545/token.txt') as f:
    token = f.readline().strip()
# Use another file to store your github username
with open('/content/gdrive/MyDrive/ece5545/git_username.txt') as f:
    handle = f.readline().strip()

In [3]:
# Clone your github repo
YOUR_TOKEN = token
YOUR_HANDLE = handle
BRANCH = "main"

%mkdir /content/gdrive/MyDrive/ece5545
%cd /content/gdrive/MyDrive/ece5545
!git clone https://{YOUR_TOKEN}@github.com/ML-HW-SYS/a4-{YOUR_HANDLE}.git
%cd /content/gdrive/MyDrive/ece5545/a4-{YOUR_HANDLE}
!git checkout {BRANCH}
!git pull

PROJECT_ROOT = f"/content/gdrive/MyDrive/ece5545/a4-{YOUR_HANDLE}"

mkdir: cannot create directory ‘/content/gdrive/MyDrive/ece5545’: File exists
/content/gdrive/MyDrive/ece5545
fatal: destination path 'a4-ZeqiGu' already exists and is not an empty directory.
/content/gdrive/MyDrive/ece5545/a4-ZeqiGu
M	src/conv2d.py
Already on 'main'
Your branch is up to date with 'origin/main'.
Already up to date.


In [4]:
# This extension reloads all imports before running each cell
%load_ext autoreload
%autoreload 2

Verify the following cell prints your github repository.

In [5]:
!ls {PROJECT_ROOT}

1-numerical_precision.ipynb  data      README.md  tests
2-svd_rank.ipynb	     mnist.py  src


In [6]:
!pip install torch numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2. Convolution

In the following cell(s), please plot the reconstruction error of an
approximated tensor (in the y-axis) with the numerical precision
(in the x-axis). Please show one plot for `winograd` and one plot for `fft`.

In [7]:
from src.conv2d import im2col
import torch
import pytest
import numpy as np
from src.conv2d import conv2d
import torch.nn.functional as F


################
# Conv2D Tests #
################
CONV2D_MAX_INPUT_SIZE = 50
CONV2D_NUM_RUNS = 100


@pytest.mark.parametrize('method_eps', [
    ('naive', 1e-4),
    ('im2col', 1e-4),
    ('fft', 1e-2)
])
@pytest.mark.parametrize('input_size', list(range(3, CONV2D_MAX_INPUT_SIZE, 4)))
def test_conv2d_largek(method_eps, input_size):
    precision = torch.float32
    method, eps = method_eps
    def round_up_to_odd(f):
        return int(np.ceil(f) // 2 * 2 + 1)

    loss_lst = []
    for seed in range(CONV2D_NUM_RUNS):
        torch.random.manual_seed(seed)
        input_size = round_up_to_odd(input_size)

        x = torch.randn((input_size, input_size), dtype=precision)
        k = torch.randn(size=(
            round_up_to_odd(input_size // 2),
            round_up_to_odd(input_size // 2)),
            dtype=precision)
        b = torch.randn(size=(1,), dtype=precision)

        ans = conv2d(x, k, b, method='torch').float()
        out = conv2d(x, k, b, method=method).float()
        assert ans.shape == out.shape, \
            "Shape mismatch. expected=%s output=%s" % (str(ans.shape), str(out.shape))
        loss_lst.append(F.l1_loss(ans, out).item())

    loss_avg = np.array(loss_lst).mean()
    assert loss_avg < eps, "Method:%s L1-error: %.4e (eps:%.2e)" \
                           % (method, loss_avg, eps)


@pytest.mark.parametrize('method_eps', [
    ('naive', 1e-4),
    ('im2col', 1e-4),
    ('winograd', 1e-3),
    ('fft', 1e-2)
])
@pytest.mark.parametrize('input_size', list(range(3, CONV2D_MAX_INPUT_SIZE, 4)))
def test_conv2d_3x3k(method_eps, input_size):
    precision = torch.float32
    method, eps = method_eps
    def round_up_to_odd(f):
        return int(np.ceil(f) // 2 * 2 + 1)

    loss_lst = []
    for seed in range(CONV2D_NUM_RUNS):
        torch.random.manual_seed(seed)
        input_size = round_up_to_odd(input_size)

        x = torch.randn((input_size, input_size), dtype=precision)
        k = torch.randn(size=(3, 3), dtype=precision)
        b = torch.randn(size=(1,), dtype=precision)

        ans = conv2d(x, k, b, method='torch').float()
        out = conv2d(x, k, b, method=method).float()
        assert ans.shape == out.shape, \
            "Shape mismatch. expected=%s output=%s" % (str(ans.shape), str(out.shape))
        loss_lst.append(F.l1_loss(ans, out).item())

    loss_avg = np.array(loss_lst).mean()
    assert loss_avg < eps, "Method:%s L1-error: %.4e (eps:%.2e)" \
                           % (method, loss_avg, eps)

In [17]:
test_conv2d_largek(('im2col', 1e-4), 31)

In [20]:
from src.conv2d import winograd
test_conv2d_3x3k(('winograd', 1e-3), 31)
# TODO: plot the error v.s. precision curve

In [14]:
from src.conv2d import fft

test_conv2d_3x3k(('fft', 1e-2), 3)
# test_conv2d_largek(('fft', 1e-2), 11)
# TODO: plot the error v.s. precision curve

result tensor([[-0.6216]])
result tensor([[-0.3724]])
result tensor([[-0.4090]])
result tensor([[0.7851]])
result tensor([[-1.1058]])
result tensor([[0.4377]])
result tensor([[2.1371]])
result tensor([[0.2469]])
result tensor([[0.1714]])
result tensor([[0.0486]])
result tensor([[-1.1457]])
result tensor([[-0.7058]])
result tensor([[0.0180]])
result tensor([[-0.0259]])
result tensor([[-0.5250]])
result tensor([[1.0310]])
result tensor([[-0.0343]])
result tensor([[-0.0135]])
result tensor([[1.5631]])
result tensor([[0.0033]])
result tensor([[-0.0120]])
result tensor([[0.0082]])
result tensor([[1.3562]])
result tensor([[1.2253]])
result tensor([[-0.7913]])
result tensor([[1.7159]])
result tensor([[0.7076]])
result tensor([[0.2823]])
result tensor([[-0.9161]])
result tensor([[1.2947]])
result tensor([[-0.0118]])
result tensor([[0.2031]])
result tensor([[0.3921]])
result tensor([[-0.5718]])
result tensor([[0.5522]])
result tensor([[0.6624]])
result tensor([[-0.7146]])
result tensor([[-0.003

AssertionError: Method:fft L1-error: 2.1913e+00 (eps:1.00e-02)

# 3. Matrix-matrix Multiply

In the following cell(s), please plot the reconstruction error (in the y-axis)
with the different numerical precisions (in the x-axis) for `log` (i.e.
logorithmic matrix-matrix multiplication).

In [ ]:
from src.matmul import logmatmul

# TODO: plot the error v.s. precision curve

